In [38]:
import sys
if '/home/ec2-user/sagemaker-pipe-template' not in sys.path:
    sys.path.insert(0, '/home/ec2-user/sagemaker-pipe-template')
import utils
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, root_mean_squared_error
import pandas as pd
import sagemaker, boto3, os
import paths, utils
from utils import Sql, train_val_test_split, get_sm_service_role_arn
import sagemaker.core.helper.session_helper as sm_session_helper
import sagemaker.core.inputs as sm_inputs
import sagemaker.core.image_uris as sm_image_uris
import sagemaker.train.model_trainer as sm_model_trainer
from sagemaker.core.training import configs as sm_train_configs
import sagemaker.core.shapes as sm_shapes


In [39]:
# User vars
region = 'us-east-1'
model_package_group_name='abalone'
model_version=1
target_name='rings'
prediction_name=target_name+'_prediction'

p=paths.Paths(bucket_name='omm-test-bucket', project_name='abalone', model_prefix='abalone')
role = get_sm_service_role_arn()
boto_session=boto3.Session(region_name=region)
sagemaker_session = sm_session_helper.Session(boto_session=boto_session)

[06/14/26 23:12:46] INFO     Found credentials from IAM      ]8;id=3572027;file:///home/ec2-user/.local/lib/python3.9/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=3572028;file:///home/ec2-user/.local/lib/python3.9/site-packages/botocore/credentials.py#1171\1171]8;;\
                             Role: ec2-1                                        


In [40]:
# DATA INGESTION
sql=Sql('user-1', 'password', 'db_1')
abalone_df = sql.query('SELECT * FROM abalone;')
abalone_df=abalone_df.drop(columns=['id','created_at'])

In [41]:
# FEATURE ENGINEERING
abalone_df = pd.get_dummies(abalone_df, columns=['sex'], drop_first=False)
abalone_df[['sex_I', 'sex_M', 'sex_F']] = abalone_df[['sex_I', 'sex_M', 'sex_F']].astype(int)
abalone_df.head()

,length,diameter,height,whole_weight,shucked_weight,viscera_weight,shell_weight,rings,sex_F,sex_I,sex_M
0,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15,0,0,1
1,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7,0,0,1
2,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9,1,0,0
3,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10,0,0,1
4,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7,0,1,0


In [ ]:
# DATA FORMATTING
y = abalone_df['rings']
X = abalone_df.drop(columns=['rings'])

X_train, X_val, X_test, y_train, y_val, y_test = train_val_test_split(X, y, train_size=0.7, val_size=0.15, random_state=42)

pd.concat([y_train, X_train], axis=1).to_csv(p.train_file, index=False, header=False)
pd.concat([y_val, X_val], axis=1).to_csv(p.validation_file, index=False, header=False)
pd.concat([y_test, X_test], axis=1).to_csv(p.test_file, index=False, header=False)
pd.concat([y_val, X_val], axis=1).to_csv(p.baseline_file, index=False, header=True) # need header

# For development and testing
X_test.to_csv(p.test_X_file, index=True, header=False)
y_test.to_csv(p.test_y_file, index=True, header=False)

input_data = {
    'train': sm_inputs.TrainingInput(p.train_dir+'/', content_type='text/csv'),
    'validation': sm_inputs.TrainingInput(p.validation_dir+'/', content_type='text/csv'),
    # xgboost only accepts train and validation 'test': sm_inputs.TrainingInput(test_s3_dir+'/', content_type='text/csv')
}

input_data_config = [
    sm_train_configs.InputData(channel_name='train', data_source=p.train_dir),
    sm_train_configs.InputData(channel_name='validation', data_source=p.validation_dir)
]

In [36]:
model_trainer = sm_model_trainer.ModelTrainer(
    training_image=sm_image_uris.retrieve('xgboost', region, version='1.5-1'),
    hyperparameters={
        'objective': 'reg:squarederror',
        'num_round': '100',
        'max_depth': '5',
        'eta': '0.2',
        'subsample': '0.8',
        'eval_metric': 'rmse'
    },
    role=role,
    sagemaker_session=sagemaker_session,
    base_job_name='abalone-train-job',
    output_data_config=sm_shapes.OutputDataConfig(s3_output_path=p.model_dir),
    networking=sm_train_configs.Networking(
        subnets=['subnet-001be661bcef4b615', 'subnet-003ad32933ca43e74'],
        security_group_ids=['sg-00f14515abe1e47e8']
    )
)

[06/14/26 23:05:54] INFO     Ignoring unnecessary instance     ]8;id=3571871;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=3571872;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/image_uris.py#535\535]8;;\
                             type: None.                                        


                    INFO     Compute not provided. Using         ]8;id=3571877;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=3571878;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/train/defaults.py#102\102]8;;\
                             default:                                           
                             instance_type='ml.m5.xlarge'                       
                             instance_count=1                                   
                             volume_size_in_gb=30                               
                             volume_kms_key_id=None                             
                             keep_alive_period_in_seconds=None                  
                             instance_groups=None                               
                             training_plan_arn=None                             
                             instance_placement_con

In [37]:
training_job = model_trainer.train(
    input_data_config=input_data_config,
    wait=True,
    logs=True
)

[06/14/26 23:05:57] INFO     SageMaker Python SDK will  ]8;id=3571895;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=3571896;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/telemetry/telemetry_logging.py#110\110]8;;\
                             collect telemetry to help                          
                             us better understand our                           
                             user's needs, diagnose                             
                             issues, and deliver                                
                             additional features.                               
                             To opt out of telemetry,                           
                             please disable via                                 
                             TelemetryOptOut parameter                          
               

FailedStatusError: Encountered unexpected failed state while waiting for TrainingJob. Final Resource State: Failed. Failure Reason: AlgorithmError: framework error: 
Traceback (most recent call last):
  File "/miniconda3/lib/python3.8/site-packages/sagemaker_containers/_trainer.py", line 84, in train
    entrypoint()
  File "/miniconda3/lib/python3.8/site-packages/sagemaker_xgboost_container/training.py", line 102, in main
    train(framework.training_env())
  File "/miniconda3/lib/python3.8/site-packages/sagemaker_xgboost_container/training.py", line 98, in train
    run_algorithm_mode()
  File "/miniconda3/lib/python3.8/site-packages/sagemaker_xgboost_container/training.py", line 64, in run_algorithm_mode
    sagemaker_train(
  File "/miniconda3/lib/python3.8/site-packages/sagemaker_xgboost_container/algorithm_mode/train.py", line 198, in sagemaker_train
    train_dmatrix, val_dmatrix, train_val_dmatrix = get_validated_dmatrices(
  File "/miniconda3/lib/python3.8/site-packages/sagema

In [33]:
model_trainer = sm_model_trainer.ModelTrainer(
    training_mode=sm_model_trainer.Mode.SAGEMAKER_TRAINING_JOB,
    training_image=sm_image_uris.retrieve('xgboost', region, version='1.5-1'),
    hyperparameters = {
        'objective': 'reg:squarederror',  # for regression (abalone rings)
        'num_round': '100',               # REQUIRED - number of boosting rounds
    },
    role=role,
    instance_count=1,
    instance_type='ml.m5.xlarge',
    output_data_config=sm_shapes.OutputDataConfig(s3_output_path=p.model_dir),
    networking=sm_train_configs.Networking(
        subnets=['subnet-001be661bcef4b615', 'subnet-003ad32933ca43e74'],
        security_group_ids=['sg-00f14515abe1e47e8']
    ),
    input_data_config=sm_train_configs.InputData('train')
    sagemaker_session=sagemaker_session
)

SyntaxError: invalid syntax (1786695786.py, line 17)

In [ ]:



# # TRAINING
# estimator = sagemaker.estimator.Estimator(
#     image_uri=sagemaker.image_uris.retrieve('xgboost', region, version='1.5-1'),
#     role=role,
#     instance_count=1,
#     instance_type='ml.m5.xlarge',
#     output_path=model_dir_uri,
#     sagemaker_session=sagemaker_session,
#     subnets=['subnet-001be661bcef4b615', 'subnet-003ad32933ca43e74'],
#     security_group_ids=['sg-00f14515abe1e47e8']
# )

# Set hyperparameters
estimator.set_hyperparameters(
    objective='reg:squarederror',
    num_round=100
)

# Train
estimator.fit(input_data)
print(estimator.model_data)

/home/ec2-user/.local/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)
INFO:sagemaker:Creating training-job with name: sagemaker-xgboost-2026-05-21-17-07-18-342


2026-05-21 17:07:23 Starting - Starting the training job...
2026-05-21 17:07:41 Starting - Preparing the instances for training...
2026-05-21 17:08:00 Downloading - Downloading input data...
2026-05-21 17:08:26 Downloading - Downloading the training image...
2026-05-21 17:09:12 Training - Training image download completed. Training in progress../miniconda3/lib/python3.8/site-packages/xgboost/compat.py:36: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  from pandas import MultiIndex, Int64Index
[2026-05-21 17:09:22.455 ip-172-31-108-128.ec2.internal:7 INFO utils.py:28] RULE_JOB_STOP_SIGNAL_FILENAME: None
[2026-05-21 17:09:22.476 ip-172-31-108-128.ec2.internal:7 INFO profiler_config_parser.py:111] User has disabled profiler.
[2026-05-21:17:09:22:INFO] Imported framework sagemaker_xgboost_container.training
[2026-05-21:17:09:22:INFO] Failed to parse hyperparameter objective value reg

In [ ]:
# BASELINING
# get baseline X
baseline=pd.read_csv(p.baseline_file, header=0)
baseline[target_name] = baseline[target_name].astype(float)
baseline_X = baseline.drop(columns=[target_name])
baseline_X.to_csv(p.baseline_X_file, index=False, header=False)

# Get predictions from baseline X
transformer = estimator.transformer(instance_count=1, instance_type='ml.m5.xlarge', output_path=p.temp_data_dir)
transformer.transform(data=p.baseline_X_file, content_type='text/csv', split_type='Line')
transformer.wait()

# Rename/move predictions file
utils.move_s3_file(boto_session, p.baseline_model_out_file, p.baseline_pred_file)

INFO:sagemaker:Creating model with name: sagemaker-xgboost-2026-05-21-17-13-20-923
INFO:sagemaker:Creating transform job with name: sagemaker-xgboost-2026-05-21-17-13-21-446


............................../miniconda3/lib/python3.8/site-packages/xgboost/compat.py:36: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  from pandas import MultiIndex, Int64Index
[2026-05-21:17:18:27:INFO] No GPUs detected (normal if no gpus installed)
[2026-05-21:17:18:27:INFO] No GPUs detected (normal if no gpus installed)
[2026-05-21:17:18:27:INFO] nginx config: 
worker_processes auto;
daemon off;
pid /tmp/nginx.pid;
error_log  /dev/stderr;
worker_rlimit_nofile 4096;
events {
  worker_connections 2048;
}
http {
  include /etc/nginx/mime.types;
  default_type application/octet-stream;
  access_log /dev/stdout combined;
  upstream gunicorn {
    server unix:/tmp/gunicorn.sock;
  }
  server {
    listen 8080 deferred;
    client_max_body_size 0;
    keepalive_timeout 3;
    location ~ ^/(ping|invocations|execution-parameters) {
      proxy_set_header X-Forwarded-For $proxy_add_

,rings_prediction,rings,length,diameter,height,whole_weight,shucked_weight,viscera_weight,shell_weight,sex_F,sex_I,sex_M
0,8.144435,7.0,0.410,0.30,0.100,0.2820,0.1255,0.057,0.0875,0,1,0
1,12.470012,13.0,0.450,0.36,0.125,0.4995,0.2035,0.100,0.1700,1,0,0
2,9.323292,11.0,0.490,0.38,0.135,0.5415,0.2175,0.095,0.1900,0,0,1
3,10.996564,14.0,0.430,0.34,0.125,0.3840,0.1375,0.061,0.1460,0,1,0
4,10.479134,11.0,0.545,0.45,0.150,0.8795,0.3870,0.150,0.2625,0,0,1


In [ ]:
# Register model from estimator
model = estimator.create_model(name=f"{model_package_group_name}-{str(model_version)}")

model.create(instance_type='ml.m5.large') # temporary instance for validation

model_package = model.register(
    content_types=['text/csv'],
    response_types=['text/csv'],
    inference_instances=['ml.m5.large'],
    transform_instances=['ml.m5.large'],
    model_package_group_name='abalone',
    description='XGBoost model for abalone ring prediction',
    approval_status='PendingManualApproval'  # or 'Approved'
)

INFO:sagemaker:Creating model with name: abalone-1
